In [ ]:
GITHUB_REPO_URL = "https://github.com/your-username/your-repo.git"
REPO_DIR = "/kaggle/working/YOLOv11-pt"
PUBLIC_DIR = "/kaggle/input/object-detection-public/public"
CHECKPOINT_DIR = "/kaggle/working/models"
CONFIG = "config/hyperparameters.yaml"

In [ ]:
import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["REPO_DIR"] = REPO_DIR
os.environ["PUBLIC_DIR"] = PUBLIC_DIR
os.environ["CHECKPOINT_DIR"] = CHECKPOINT_DIR
os.environ["CONFIG"] = CONFIG

In [ ]:
# Xoa thu muc repo cu trong /kaggle/working neu notebook duoc chay lai.
!rm -rf "$REPO_DIR"

# Clone codebase moi nhat tu GitHub vao /kaggle/working.
!git clone "$GITHUB_REPO_URL" "$REPO_DIR"

# Chuyen working directory cua notebook vao repo vua clone.
%cd $REPO_DIR

In [ ]:
# Cai dat cac dependency Python duoc khai bao trong codebase.
!python -m pip install -q -r requirements.txt

In [ ]:
# Chay smoke test cho compute_ap, compute_metric, compute_ciou va non_max_suppression.
!python -B smoke_metrics.py

In [ ]:
# Train model ResNet50 pretrained + legacy YOLO FPN/head, validate moi 5 epoch va luu checkpoint.
!python train.py --config "$CONFIG" --train_data "$PUBLIC_DIR/annotations/train.json" --val_data "$PUBLIC_DIR/annotations/val.json" --image_dir "$PUBLIC_DIR/train/images" --val_image_dir "$PUBLIC_DIR/val/images" --checkpoint_dir "$CHECKPOINT_DIR"

In [ ]:
# Chay inference tren tap validation bang checkpoint tot nhat.
!python predict.py --config "$CONFIG" --checkpoint "$CHECKPOINT_DIR/best.pth" --image_dir "$PUBLIC_DIR/val/images" --output "/kaggle/working/val_predictions.json"

In [ ]:
# Danh gia predictions bang evaluator duoc cung cap trong public/tools.
!python public/tools/evaluate_predictions.py --ground_truth "$PUBLIC_DIR/annotations/val.json" --predictions "/kaggle/working/val_predictions.json" --output "/kaggle/working/val_score.json"

In [ ]:
# Copy cac artifact quan trong ra /kaggle/working de Kaggle luu lai sau khi notebook ket thuc.
!cp "$CHECKPOINT_DIR/best.pth" /kaggle/working/best.pth
!cp "$CHECKPOINT_DIR/last.pth" /kaggle/working/last.pth
!cp "$CHECKPOINT_DIR/train_log.csv" /kaggle/working/train_log.csv

# Nen checkpoint, log va ket qua validation thanh mot file zip de tai ve de hon.
!zip -j /kaggle/working/submission_artifacts.zip /kaggle/working/best.pth /kaggle/working/last.pth /kaggle/working/train_log.csv /kaggle/working/val_predictions.json /kaggle/working/val_score.json